# Bistable Mass-in-Mass Simulation

### Imports


In [ ]:
import jax.numpy as jnp
from jax import grad, jit, vmap
from jax.experimental.ode import odeint
from jax import config
config.update("jax_enable_x64", True)  # enable float64 type
from pathlib import Path
from matplotlib import animation
from matplotlib import cm, colors
import matplotlib.animation as animation
from matplotlib.collections import (LineCollection, PolyCollection)
import matplotlib
import matplotlib.pyplot as plt
from jax import grad, jit

### Define fourth order energy potential

In [ ]:
def potential_order4(ks, x):
    'ks = k_fit4'
    'x = d2-d1'
    k2 = ks[0]
    k4 = ks[1]

    k3 = 3*jnp.sqrt(k2*k4/2)

    return k2/2*(x)**2 - k3/3*(x)**3 + k4/4*(x)**4

def potential_order4_summed(ks, x):
    'ks = k_fit4'
    'x = d2-d1'
    k2 = ks[0]
    k4 = ks[1]

    k3 = 3*jnp.sqrt(k2*k4/2)

    return jnp.sum(k2/2*(x)**2 - k3/3*(x)**3 + k4/4*(x)**4, axis=-1)


# Parameters fitted to experimental data
k2 = 300
k4 = 5386828
k3 = 3*jnp.sqrt(k2*k4/2) # ensures a symmetric potential well
k_fit4 = jnp.array([k2, k4])

x_eq2 = k3/(2*k4) + jnp.sqrt( (k3/(2*k4))**2 - k2/k4 )
x_eqs_O4 = jnp.array([0, x_eq2]) # equilibrium positions (i.e., displacement value at energy minimas)


d1 = 0 # displacement of outer mass
d2 = jnp.linspace(-0.004, 0.015, 1000) # displacement of inner mass
fitted_energy4 = potential_order4(k_fit4, d2)

# Visualize the energy landscape
plt.figure(figsize=[4,3])
plt.plot(d2*10**3, fitted_energy4*10**3, 'k')
plt.xlabel('Displacement (mm)')
plt.ylabel('Energy (mJ)')
plt.title('Bistable Energy Landscape')
plt.tight_layout()

### Set up the simulation

In [ ]:
# Define function to return local and global states based on constraints
def get_local_global_states(n_dof, m1_state0, m2_state0, m1_constrained_unit_ids, m2_constrained_unit_ids):

    # Define global state indices as <u_0, v_0, u_1, v_1, u_2, v_2, .... u_n, v_n>
    state0_global = jnp.zeros([2, n_dof])
    state0_global = state0_global.at[:, 0::2].set(m1_state0)
    state0_global = state0_global.at[:, 1::2].set(m2_state0)

    if len(m1_constrained_unit_ids)>0:
        m1_ad_free_dof_ids = jnp.delete(jnp.arange(0, n_dof, 2), m1_constrained_unit_ids)
    else:
        m1_ad_free_dof_ids = jnp.arange(0, n_dof, 2)
    if len(m2_constrained_unit_ids)>0:
        m2_ad_free_dof_ids = jnp.delete(jnp.arange(1, n_dof, 2), m2_constrained_unit_ids)
    else:
        m2_ad_free_dof_ids = jnp.arange(1, n_dof, 2)


    state0_local = jnp.zeros([2, len(m1_ad_free_dof_ids)+len(m2_ad_free_dof_ids)])
    state0_local = state0_local.at[:, 0::2].set(state0_global[:, m1_ad_free_dof_ids])
    state0_local = state0_local.at[:, 1::2].set(state0_global[:, m2_ad_free_dof_ids])

    return state0_local, state0_global

# Define function to return local and global states based on constraints and phase
def get_current_local_global_states(phase_description, n_units, x_eq2s, m1_state0, m1_constrained_unit_ids, m2_constrained_unit_ids):
    '''
    gives updated local and global state with updated phase description
    '''
    n_dof = 2*n_units
    p2_units = []
    for i, unit_phase in enumerate(phase_description):
        if unit_phase == '1':
            p2_units.append(i)

    m2_state0_p2 = jnp.zeros([2, n_units]) # displacement, velocity starting in phase 2
    if len(p2_units)>0:
        p2_unit_ids = jnp.asarray(p2_units) + 1
        m2_state0_p2 = m2_state0_p2.at[0, p2_unit_ids].set(x_eq2s)
    state0_local_, state0_global_ = get_local_global_states(n_dof, m1_state0, m2_state0_p2, m1_constrained_unit_ids, m2_constrained_unit_ids)

    return state0_local_, state0_global_

def get_model_info(potential_model_name, n_units):

    'potential_model_name: (str) --> "4th Order"'
    'n_units: (int) --> number of units in the simulation (including fixed unit at beginning and end)'

    if potential_model_name == '4th Order':
          # Potential function
          potential_fn = potential_order4
          potential_summed = potential_order4_summed

          # Potential parameters
          bistable_potential_params = jnp.zeros([n_units, k_fit4.shape[0]])
          bistable_potential_params = bistable_potential_params.at[1:-1,:].set(k_fit4*jnp.ones_like(bistable_potential_params.shape[0]))

          # Equilibrium points
          equilibrium1 = x_eqs_O4[0]
          equilibrium2 = x_eqs_O4[1]
    
    return potential_fn, potential_summed, bistable_potential_params, equilibrium1, equilibrium2

def get_free_dof_parameters(m1s, m2s, c1s, c2s, mu_ks, stiffness_vals, m1_constrained_ids, m2_constrained_ids):

    if len(m1_constrained_ids)>0:
        m1s_free = jnp.delete(m1s, m1_constrained_ids)
        c1s_free = jnp.delete(c1s, m1_constrained_ids)
        mu_ks_free = jnp.delete(mu_ks, m1_constrained_ids)
    else:
        m1s_free = m1s
        c1s_free = c1s
        mu_ks_free = mu_ks
    if len(m2_constrained_ids)>0:
        m2s_free = jnp.delete(m2s, m2_constrained_ids)
        c2s_free = jnp.delete(c2s, m2_constrained_ids)
        stiffness_vals_free = jnp.delete(stiffness_vals, m2_constrained_ids, axis=0)
    else:
        m2s_free = m2s
        c2s_free = c2s
        stiffness_vals_free = stiffness_vals

    return m1s_free, m2s_free, c1s_free, c2s_free, mu_ks_free, stiffness_vals_free

# Define plotting function
def plot_response(num_solution, timepoints, alpha=1, cmap_temporal='Dark2', sup_title=None, figsize=[8, 5]):
    'Plot spatiotemporal plots and temporal signals of un and vn-un of simulations'
    def cmap_to_color_list(cmap_name, n_discrete_regions):
        cmap = cm.get_cmap(cmap_name, n_discrete_regions)

        color_list = []
        for i in range(cmap.N):
            rgba = cmap(i)
            color_list.append(matplotlib.colors.rgb2hex(rgba))
        
        return color_list

    n_units = num_solution.shape[2] - 2

    if cmap_temporal != 'Dark2':
        colors = cmap_to_color_list(cmap_temporal, n_units)
    else:
        colors = cmap_to_color_list(cmap_temporal, 8)

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=figsize)

    orig_map=plt.cm.get_cmap('plasma')
    rev_cmap = orig_map.reversed()

    data_set = num_solution*10**(3)

    un_disps = data_set[:, 0, 1:-1]
    vn_disps = data_set[:, 2, 1:-1]

    p1 = ax1.pcolor(un_disps.T, cmap=rev_cmap)
    p2 = ax2.pcolor((vn_disps - un_disps).T, cmap=rev_cmap)

    plt.colorbar(p1, ax=ax1, pad=0.01, label='$u_n$ (mm)')
    plt.colorbar(p2, ax=ax2, pad=0.01, label='$v_n-u_n$ (mm)')

    ax1.set_title('Displacement, $u_n$', fontsize=12)
    ax2.set_title('Displacement, $v_n - u_n$', fontsize=12)

    ax1.set_yticks(jnp.arange(0.5, 0.5+un_disps.shape[1]))
    ax1.set_yticklabels(jnp.arange(un_disps.shape[1]), fontsize=12)
    ax2.set_yticks(jnp.arange(0.5, 0.5+un_disps.shape[1]))
    ax2.set_yticklabels(jnp.arange(un_disps.shape[1]), fontsize=12)

    ax1.set_xticks(jnp.array([0, 500, 1000]))
    ax1.set_xticklabels(jnp.array([0, .5, 1]), fontsize=12)
    ax2.set_xticks(jnp.array([0, 500, 1000]))
    ax2.set_xticklabels(jnp.array([0, .5, 1]), fontsize=12)


    ax1.set_xlabel('Time (s)', fontsize=12)
    ax2.set_xlabel('Time (s)', fontsize=12)

    ax1.set_ylabel('$n$', fontsize=12)
    ax2.set_ylabel('$n$', fontsize=12)

    for i in jnp.arange(n_units):
        ax3.plot(timepoints, un_disps[:, i], color=colors[i], alpha=alpha, label=f'$u_{i}$')

        ax4.plot(timepoints, (vn_disps[:, i]-un_disps[:, i]), color=colors[i], alpha=alpha, label=f'$v_{i}-u_{i}$')
    
    ax3.legend(fontsize=6, loc='upper right')
    ax4.legend(fontsize=6, loc='upper right')

    ax3.set_xlim([0, max(T)])
    ax4.set_xlim([0, max(T)])
    
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Displacement (mm)')
    ax3.set_title('$u_n$ (mm)')
    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('Displacement (mm)')
    ax4.set_title('$v_n-u_n$ (mm)')

    if sup_title is not None:
        plt.suptitle(f'{sup_title}')
        
    plt.tight_layout()



In [ ]:
# Define mass profile in the chain
n_units = 3
n_units+=2 # account for constrained ends

# Define Physical parameters
a = 40*10**(-3) # unit cell length, [m]
m1 = 40*10**(-3) # [kg]
m2 = 15*10**(-3) # [kg]
k1 = 2000 # [N/m]
c1 = 0.4 # [kg/s]
c2 = 0.9 # [kg/s]
mu_k = 0 # coefficient of kinetic friction
potential_model_name = '4th Order' # 4th order is the only one implemented in this notebook

potential_fn, potential_summed, bistable_potential_params, x_eq1s, x_eq2s = get_model_info(potential_model_name, n_units)
stiffness_vals = jnp.concatenate([k1*jnp.ones([n_units,1]), bistable_potential_params], axis=-1)

# Define state variables
m1_state0 = jnp.zeros([2, n_units]) # displacement, velocity
m2_state0 = jnp.zeros([2, n_units]) # displacement, velocity starting in phase 1
m1_state0 = m1_state0.at[0, :].set(0.00000000001)

n_dof = 2*n_units # Number of degrees of freedom. Each unit has 2 dof

# Define constrained unit ids
m1_constrained_unit_ids = jnp.array([0, -1]) # First unit is driven, last unit is fixed
m2_constrained_unit_ids = jnp.array([0, -1]) # Inner mass of unit is driven, inner mass of last unit is fixed
constrained_unit_ids = (m1_constrained_unit_ids, m2_constrained_unit_ids)

# Define zero displacement for constrained units
def constsignal(t):
     return 0
def dconstsignal(t):
     return 0

# Reshape the local state variable to global state variable
def reshape_local_to_global(free_dof_solution, t, timepoints, impulse_fn_data, amplitude_impulse_scaling=1.):
     global_solution = jnp.zeros([timepoints.shape[0], 4, n_units])

     def impulse_fn(t_):
          return jnp.interp(t_, timepoints, amplitude_impulse_scaling*impulse_fn_data)

     def dimpulse_fn(t_):
          ds = grad(impulse_fn)
          return ds(t_)

     global_solution = global_solution.at[:, 0, 0].set(impulse_fn(t)) # applied displacement
     global_solution = global_solution.at[:, 1, 0].set(vmap(dimpulse_fn)(t)) # applied velocity
     global_solution = global_solution.at[:, 2, 0].set(impulse_fn(t)) # applied displacement
     global_solution = global_solution.at[:, 3, 0].set(vmap(dimpulse_fn)(t)) # applied velocity

     # displacement values
     global_solution = global_solution.at[:, 0, 1:-1].set(free_dof_solution[:, 0, 0::2]) # free m1 dof disp solution
     global_solution = global_solution.at[:, 2, 1:-1].set(free_dof_solution[:, 0, 1::2]) # free m2 dof disp solution
     # velocity values
     global_solution = global_solution.at[:, 1, 1:-1].set(free_dof_solution[:, 1, 0::2]) # free m1 dof vel solution
     global_solution = global_solution.at[:, 3, 1:-1].set(free_dof_solution[:, 1, 1::2]) # free m2 dof disp solution
     
     return global_solution

# Define potential energy of the constrained spring-mass system
def potential_energy_constrained_ends(free_dof_displacement, t, stiffness_vals, impulse_fn):
     "free_dof_displacement: <1xn_free_dof> assumes u_0, u_-1, v_-1 are all constrained --> jnp.array([u_1, v_1, u_2, v_2, u_3, v_3, ...., u_-2, v_-2])"
     "t: time"
     "stiffness_vals: jnp.array([k1, k2, k3, k4,...]) used to calculate the coupling potential"
     "impulse_fn: imposed displacement (impulse1(t), impulse2(t), gaussian_impulse(t), etc.)"

     displacement1 = jnp.zeros(n_units)
     displacement2 = jnp.zeros(n_units)

     displacement1 = displacement1.at[0].set(impulse_fn(t)) # impose applied displacement
     displacement2 = displacement2.at[0].set(impulse_fn(t)) # impose applied displacement

     displacement1 = displacement1.at[-1].set(constsignal(t)) # impose fixed end
     displacement2 = displacement2.at[-1].set(constsignal(t)) # impose fixed end

     displacement1 = displacement1.at[1:-1].set(free_dof_displacement[0::2]) # set the free dofs
     displacement2 = displacement2.at[1:-1].set(free_dof_displacement[1::2]) # set the free dofs

     k1, k2, k4 = stiffness_vals.T
     potential_energy_constr = 0.5 * jnp.sum(k1[0] * jnp.diff(displacement1)**2, axis=-1) + jnp.sum(potential_fn(jnp.array([k2, k4]), displacement2-displacement1), axis=-1)

     return potential_energy_constr

# Calculate the force as the gradient of the potential energy
force_constrained_ends = grad(lambda free_dof_displacement, t, stiffness_vals, impulse_fn: -potential_energy_constrained_ends(free_dof_displacement, t, stiffness_vals, impulse_fn), argnums=0)

potential_energy_fn = potential_energy_constrained_ends
force_fn = force_constrained_ends

# Define RHS
@jit
def rhs(state, t, timepoints, m1s, m2s, stiffness_vals, c1s, c2s, mu_ks, impulse_fn_data):
     "state: local state0 < 2 x n_free_dof >"
     "t: time"
     "timepoints: discrete timepoints"
     "m1: mass 1 (n_units, 1)"
     "m2: mass 2 (n_units, 1)"
     "stiffness_vals: (n_units, len(bistable_potential_params)+1) to be passed to force_constr..."
     "c1s: c1 damping (n_units, 1)"
     "cc2: c2 damping (n_units, 1)"
     "mu_k: friction (n_units, 1)"
     "impulse_fn_data: impulse_fn(timepoints)"
     
     displacement, velocity = state
     
     def impulse_fn(t_):
          return jnp.interp(t_, timepoints, impulse_fn_data)
     
     mass_vals = jnp.ones_like(state[0])
     mass_vals = mass_vals.at[0::2].set(m1s)
     mass_vals = mass_vals.at[1::2].set(m2s) # array of m2, m1, m2, m1, m2

     damping_un = c1s*(velocity[0::2]) + (mu_ks)*9.81*m1s*jnp.sign(velocity[0::2])
     damping_vn = jnp.zeros_like(velocity[1::2])
     damping_vn = damping_vn.at[:].set(c2s*(velocity[1::2] - velocity[0::2]))

     damping = jnp.zeros_like(velocity)
     damping = damping.at[0::2].set(damping_un)
     damping = damping.at[1::2].set(damping_vn)

     acceleration = (force_fn(displacement, t, stiffness_vals, impulse_fn) - damping)/mass_vals

     return jnp.array([velocity, acceleration])


def twogaussian_impulse(t, A, t_s, w, t_s2_factor=1.5):
    """
    t: timepoints (array)
    A: amplitude [m] (float)
    t_s: time_shift (float)
    w: width of pulse (float)
    t_s2_factor: scaling factor for impulse #2. defaults to 1.5 (float)
    """

    return A*jnp.exp(-(t-t_s)**2 / w**2)+A*jnp.exp(-(t-t_s2_factor*t_s)**2 / w**2)

def gaussian_impulse(t, A, t_s, w):
    """
    t: timepoints (array)
    A: amplitude [m] (float)
    t_s: time_shift (float)
    w: width of pulse (float)"""

    return A*jnp.exp(-(t-t_s)**2 / w**2)

In [ ]:
# Define arrays of variables to pass into rhs function. Currently, all unit cells are identical.
m1s = m1*jnp.ones([n_units])
m2s = m2*jnp.ones([n_units])
c1s = c1*jnp.ones([n_units])
mu_ks = mu_k*jnp.ones([n_units])
c2s = c2*jnp.ones([n_units])

m1s_free, m2s_free, c1s_free, c2s_free, mu_ks_free, stiffness_vals_free = get_free_dof_parameters(m1s, m2s, c1s, c2s, mu_ks, stiffness_vals, m1_constrained_unit_ids, m2_constrained_unit_ids)

### Look at output of simulation for one impulse

In [ ]:
# Get the local and global states based on the initial_phase
initial_phase = '000' # Initial phase of the structure represented as bits
state0_local_, state0_global_ = get_current_local_global_states(initial_phase, n_units, -x_eq2s, m1_state0, m1_constrained_unit_ids, m2_constrained_unit_ids)

# Impulse parameters
impulse_to_model = gaussian_impulse # Name the function defined as the applied displacement
A = -6*10**(-3)
w = 0.015
t_s = 0.055
T = jnp.linspace(0, 0.45, 1000) # Time (s)

# Integrate rhs function
sol_free = odeint(rhs, state0_local_, T, T, m1s_free, m2s_free, stiffness_vals, c1s_free, c2s_free, mu_ks_free, impulse_to_model(T, A, t_s, w))
sol_global = reshape_local_to_global(sol_free, T, T, impulse_to_model(T, A, t_s, w))[:, :, :]

# Plot the impulse
plt.figure(figsize=[7,2])
plt.plot(T, impulse_to_model(T, A, t_s, w)*10**3, 'k')
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('$\\bar{u}(t)$ (mm)', fontsize=12)
plt.grid(False)
plt.xlim([0, max(T)])
plt.title('Applied Displacement')
plt.tight_layout()

# Plot the spatiotemporal plots and temporal signals
plot_response(sol_global, T, figsize=[10,5])